In [1]:
import torch, glob, re
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import numpy as np
import torch.nn.functional as F
from collections.abc import Iterable

class DINOEncoder:
	def __init__(self, device="cuda:0"):
		self.device = device
		# Load DINOv2 model (base version, 768-dim embeddings)
		# 94 vs 58 = 36; 95-45=50
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
		#        self.model = AutoModel.from_pretrained('facebook/dinov2-base').to(device)

		# 91 vs 47 = 44; 89 - 40=49
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-vitl16-pretrain-lvd1689m')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-vitl16-pretrain-lvd1689m').to(device)

		# 92 vs 59 = 33; 91 - 40=51
		self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-vits16-pretrain-lvd1689m')
		self.model = AutoModel.from_pretrained('facebook/dinov3-vits16-pretrain-lvd1689m').to(device)

		# 94 vs 64 = 30; 94 - 54=40
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-vits16plus-pretrain-lvd1689m')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-vits16plus-pretrain-lvd1689m').to(device)

		# 95 vs 58 = 37; 93 - 53=40
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-vitb16-pretrain-lvd1689m')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-vitb16-pretrain-lvd1689m').to(device)

		# 88 vs 30 = 58; 88 vs 24=64
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-vith16plus-pretrain-lvd1689m')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-vith16plus-pretrain-lvd1689m').to(device)

		# 93 vs 43 = 50; 94 - 40=54
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-convnext-large-pretrain-lvd1689m')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-convnext-large-pretrain-lvd1689m').to(device)

		# 93 vs 62 = 31; 97 - 68=29
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-convnext-tiny-pretrain-lvd1689m')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-convnext-tiny-pretrain-lvd1689m').to(device)

		# 96 vs 68 = 28; 96 - 65=31
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-convnext-small-pretrain-lvd1689m')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-convnext-small-pretrain-lvd1689m').to(device)

		# 93 vs 56 = 37; 96 - 54=42
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-convnext-base-pretrain-lvd1689m')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-convnext-base-pretrain-lvd1689m').to(device)

		# 88 vs 34; 85-27=58
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-vit7b16-pretrain-lvd1689m')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-vit7b16-pretrain-lvd1689m').to(device)

		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-vitl16-chmv2-dpt-head')
		#        self.model = AutoModel.from_pretrained('facebook/dinov3-vitl16-chmv2-dpt-head').to(device)

		# 99 vs 87; 99-88=11
		#        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-vit7b16-pretrain-sat493m')
		#       self.model = AutoModel.from_pretrained('facebook/dinov3-vit7b16-pretrain-sat493m').to(device)

		# 88 vs 34; 85 - 27=58
		#        self.processor = AutoImageProcessor.from_pretrained('mirekphd/dinov3-vit7b16-pretrain-lvd1689m-fp16')
		#        self.model = AutoModel.from_pretrained('mirekphd/dinov3-vit7b16-pretrain-lvd1689m-fp16').to(device)

		self.processor.size = {'height': 448, 'width': 448}  # 88 vs 24 with vith16plus

	# mirekphd/dinov3-vit7b16-pretrain-lvd1689m-fp16
	def encode_image(self, image):
		"""
		Encode a single image into a feature vector.

		Args:
			image: PIL Image or numpy array (BGR from AI2-THOR)

		Returns:
			torch.Tensor of shape (768,) - feature embedding
		"""
		# Convert BGR to RGB if needed
		if isinstance(image, np.ndarray):
			if image.shape[-1] == 3:
				image = Image.fromarray(image[:, :, ::-1])  # BGR to RGB
		elif not isinstance(image, Image.Image):
			image = Image.fromarray(image)

		# Process and encode
		inputs = self.processor(images=image, return_tensors='pt')
		inputs = {k: v.to(self.device) for k, v in inputs.items()}

		with torch.no_grad():
			outputs = self.model(**inputs)
			# DINOv2 outputs: last_hidden_state shape (1, 197, 768)
			# We take the [CLS] token (first token) as the image representation
			embedding = outputs.last_hidden_state[:, 0, :].squeeze()  # Shape: (768,)

		return embedding

	def encode_batch(self, images):
		"""
		Encode a batch of images.

		Args:
			images: List of PIL Images or numpy arrays

		Returns:
			torch.Tensor of shape (N, 768)
		"""
		embeddings = [self.encode_image(img) for img in images]
		return torch.stack(embeddings)

	# def encode_batch_mean(self, images):
	# 	path_embeds_cls = self.encode_batch(images)
	# 	path_embeds_mean = path_embeds_cls.mean(dim=0)
	# 	return path_embeds_mean

	def encode_batch_mean(self, images):
		# 1. Get raw [CLS] embeddings for the path (Shape: N, 768)
		path_embeds_cls = self.encode_batch(images)

		# 2. Normalize each individual frame's vector to unit length
		path_embeds_norm = F.normalize(path_embeds_cls, p=2, dim=1)

		# 3. Take the directional average of the paths
		path_embeds_mean = path_embeds_norm.mean(dim=0)

		# 4. Re-normalize the final average vector so it has a magnitude of 1.0
		# This guarantees flawless alignment with ChromaDB's internal HNSW cosine index
		final_signature = F.normalize(path_embeds_mean, p=2, dim=0)

		return final_signature

	def encode_batch_normalized(self, images):
		# 1. Get raw [CLS] embeddings for the path (Shape: N, 768)
		path_embeds_cls = self.encode_batch(images)

		# 2. Normalize each individual frame's vector to unit length
		path_embeds_norm = F.normalize(path_embeds_cls, p=2, dim=1)

		return path_embeds_norm
    
	def compare_mean_embeddings(self, mean_embeds1, mean_embeds2):
		# if we have a set of embeddings to compare, then compare all, otherwise just the one
		if isinstance(mean_embeds1, Iterable):
			return {F.cosine_similarity(me, mean_embeds2, dim=0) for me in mean_embeds1}
		else:
			return F.cosine_similarity(mean_embeds1, mean_embeds2, dim=0)

	def compare_paths(self, ref_path, cur_path):
		#(ref_path_embeds, cur_path_embeds) = self.get_embeddings(ref_path, cur_path)
		# print(ref_path_embeds)
		ref_path_embeds = self.encode_batch(ref_path)
		cur_path_embeds = self.encode_batch(cur_path)

		ideal_path_normalized = F.normalize(ref_path_embeds, dim=1)
		current_path_normalized = F.normalize(cur_path_embeds, dim=1)

		# Get similarity to all reference frames in one matrix multiplication
		similarities = torch.mm(current_path_normalized, ideal_path_normalized.t()).squeeze()

		# logits[range(len(logits)), range(len(logits[0]))] = 0 # we're not interested in each image compared to itself, so set the diagonal to 0
		# 4. Convert logits to probabilities using Softmax
		probs = F.softmax(similarities, dim=-1)
		return probs

	def compare_paths_mean(self, ref_path, cur_path):
		ref_path_embeds = self.encode_batch(ref_path)
		cur_path_embeds = self.encode_batch(cur_path)

		ref_mean = ref_path_embeds.mean(dim=0)
		cur_mean = cur_path_embeds.mean(dim=0)
		return F.cosine_similarity(ref_mean, cur_mean, dim=0)

	def extract_number(self, filename):
		# Extract the number from the filename (assuming it's the step count)
		# This regex looks for digits at the beginning, end, or between non-digits
		numbers = re.findall(r'\d+', filename)
		return int(numbers[-1]) if numbers else 0

	def load_images(self, path):
		imgs_path = glob.glob(path)
		imgs_path = sorted(imgs_path, key=self.extract_number)
		pil_images = [Image.open(fname).convert('RGB') for fname in imgs_path]
		return pil_images

	def load_path(self, base_dir):
		return self.load_images(base_dir + "/*.png")

In [2]:
de = DINOEncoder()

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [3]:
base_uri = "/home/hp20024/robotics/latent_planning/ae_semantic_navigation_client/src/ae_semantic_navigation_client/debug_door_pics/5048_*.png"
imgs = de.load_images(base_uri)
len(imgs)

40

In [4]:
def extract_number(filename):
    # Extract the number from the filename (assuming it's the step count)
    # This regex looks for digits at the beginning, end, or between non-digits
    numbers = re.findall(r'\d+', filename)
    return int(numbers[-1]) if numbers else 0
        
def load_images2(path):
    imgs_path = glob.glob(path)
    imgs_full = [ip for ip in imgs_path if '_w.png' in ip]
    imgs_doors_only = [ip for ip in imgs_path if '_w.png' not in ip]
    
    imgs_full = sorted(imgs_full, key=extract_number)
    imgs_doors_only = sorted(imgs_doors_only, key=extract_number)

    #print(imgs_full)
    #print(imgs_doors_only)
    
    pil_images_full = [Image.open(fname).convert('RGB') for fname in imgs_full]
    pil_images_doors_only = [Image.open(fname).convert('RGB') for fname in imgs_doors_only]
    
    return pil_images_full, pil_images_doors_only, imgs_full, imgs_doors_only
    #
loaded_full, loaded_doors, imgs_full_names, imgs_doors_only_names = load_images2(base_uri)

In [5]:
img_names_full = [ifn[ifn.rfind('/') + 1:] for ifn in imgs_full_names]
img_names_doors_only = [ifn[ifn.rfind('/') + 1:] for ifn in imgs_doors_only_names]
print(img_names_doors_only[7], img_names_full[7])

5048_0_65.png 5048_0_65_w.png


In [6]:
#[img.size for img in imgs]
embedded_doors = de.encode_batch_normalized(loaded_doors)
embedded_full = de.encode_batch_normalized(loaded_full)

In [7]:
F.cosine_similarity(embedded_full[1], embedded_full[18], dim=0)

tensor(0.9547, device='cuda:0')

In [8]:
contamination_threshold = 0.65
norm_embeddings = embedded_full #embedded_doors
image_list = loaded_doors

# 3. Compute the all-to-all pairwise similarity matrix 
# Shape: (N, N)
similarity_matrix = torch.mm(norm_embeddings, norm_embeddings.t())

# 4. Calculate the mean agreement score for each frame relative to the group
# We subtract 1.0 from the sum because every image has a perfect 1.0 similarity with itself
num_images = len(image_list)
mean_agreement = (similarity_matrix.sum(dim=1) - 1.0) / (num_images - 1)

# 5. Filter out frames that fall below the semantic consensus
clean_images = []
for idx, score in enumerate(mean_agreement.tolist()):
    if score >= contamination_threshold:
        clean_images.append(image_list[idx])
    else:
        print(f"[Outlier Rejected] Frame index {idx} dropped. Agreement score: {score:.4f}")
        
# Safeguard: If everything got wiped out by a strict threshold, fall back to raw data
if not clean_images:
    return image_list

[Outlier Rejected] Frame index 7 dropped. Agreement score: 0.6066
[Outlier Rejected] Frame index 12 dropped. Agreement score: 0.5276
[Outlier Rejected] Frame index 15 dropped. Agreement score: 0.5513
[Outlier Rejected] Frame index 17 dropped. Agreement score: 0.5755


SyntaxError: 'return' outside function (2147432040.py, line 24)

In [9]:
similarity_threshold = 0.60
embeddings = embedded_full #embedded_doors
num_frames = len(embeddings)
filenames = img_names_full

# 1. Stack list into a unified tensor (Shape: N, 768)
# If embeddings is already a stacked tensor, this is a zero-cost operation
if isinstance(embeddings, list):
    embeddings_tensor = torch.stack(embeddings)
else:
    embeddings_tensor = embeddings

# 2. Compute the complete all-to-all similarity matrix in one GPU cycle
# Matrix multiplication of normalized vectors yields the exact cosine similarities
sim_matrix = torch.mm(embeddings_tensor, embeddings_tensor.t())

# 3. Establish consensus: Count how many other frames each image agrees with
# We check how many values in each row cross your 0.60 requirement
agreement_mask = sim_matrix >= similarity_threshold
agreement_counts = agreement_mask.sum(dim=1) # Shape: (N,)

# 4. Filter logic: Keep frames that match the global majority
# A true frame should agree with at least 50% of the trajectory cohort
majority_cutoff = num_frames // 2

final_embeddings = []
for idx in range(num_frames):
    if agreement_counts[idx] >= majority_cutoff:
        fname = filenames[idx] if idx < len(filenames) else f"Index_{idx}"
        print(f"AE: [KEPT]: {fname} | Global agreement count: {agreement_counts[idx]}/{num_frames}")
        final_embeddings.append(embeddings_tensor[idx])
    else:
        # Safe localized filename extraction
        fname = filenames[idx] if idx < len(filenames) else f"Index_{idx}"
        print(f"AE: [Vector Purge] throwing out: {fname} | Global agreement count: {agreement_counts[idx]}/{num_frames}")

return final_embeddings

AE: [KEPT]: 5048_0_5_w.png | Global agreement count: 17/20
AE: [KEPT]: 5048_0_9_w.png | Global agreement count: 16/20
AE: [KEPT]: 5048_0_49_w.png | Global agreement count: 17/20
AE: [KEPT]: 5048_0_52_w.png | Global agreement count: 17/20
AE: [KEPT]: 5048_0_54_w.png | Global agreement count: 17/20
AE: [KEPT]: 5048_0_58_w.png | Global agreement count: 17/20
AE: [KEPT]: 5048_0_62_w.png | Global agreement count: 17/20
AE: [KEPT]: 5048_0_65_w.png | Global agreement count: 10/20
AE: [KEPT]: 5048_0_66_w.png | Global agreement count: 16/20
AE: [KEPT]: 5048_0_69_w.png | Global agreement count: 16/20
AE: [KEPT]: 5048_0_71_w.png | Global agreement count: 16/20
AE: [KEPT]: 5048_0_73_w.png | Global agreement count: 16/20
AE: [Vector Purge] throwing out: 5048_0_74_w.png | Global agreement count: 3/20
AE: [KEPT]: 5048_0_75_w.png | Global agreement count: 16/20
AE: [KEPT]: 5048_0_78_w.png | Global agreement count: 16/20
AE: [Vector Purge] throwing out: 5048_0_79_w.png | Global agreement count: 3/20
AE

SyntaxError: 'return' outside function (2615934543.py, line 37)

In [10]:
def weed_out_odd_images(embeddings):
    final_embeddings = []
    similarity_threshold = 0.60

    # handle cases with 0, 1 and 2 images only
    if len(embeddings) < 2: return embeddings
    if len(embeddings) == 2 and F.cosine_similarity(embeddings[0], embeddings[1], dim=0) >= similarity_threshold: 
        return embeddings
    elif (len(embeddings) == 2 and F.cosine_similarity(embeddings[0], embeddings[1], dim=0) < similarity_threshold):
        return final_embeddings

    # if we're here, then there's at least 3 embeddings, let's see if the 1st one is similar to any others
    sim_0_1 = F.cosine_similarity(embeddings[0], embeddings[1], dim=0)
    sim_0_2 = F.cosine_similarity(embeddings[0], embeddings[2], dim=0)
    if (sim_0_1 < similarity_threshold and sim_0_2 < similarity_threshold):
        print("AE: throwing out 0: ", img_names_full[0])
        return weed_out_odd_images(embeddings[1:]) # the first image is not a good match, let's drop it
    else: # the first image matches with either the 2nd or 3rd, the rest will be handled by a loop
        final_embeddings.append(embeddings[0])

        img1_index = 0
        img2_index = 1
        while img2_index < len(embeddings):
            img1 = embeddings[img1_index]
            img2 = embeddings[img2_index]
            cos_sim = float(F.cosine_similarity(img1, img2, dim=0))
            if cos_sim >= similarity_threshold: # img2 is goog, let's add it and move on to the next pair (img2 with img3)
                #print("AE: keeping: ", img_names_full[img2_index], " cos_sim = ", cos_sim)
                final_embeddings.append(img2)
                img1_index = img2_index
                img2_index += 1
            else: # img2 is bad, let's drop it and compare img1 with img3
                print("AE: throwing out: ", img2_index, " # " , img_names_full[img2_index], " cos_sim = ", cos_sim)
                img2_index += 1
                
    return final_embeddings

weeded_f = weed_out_odd_images(embedded_doors)
print(len(weeded_f))
print(len(embedded_doors))

AE: throwing out:  12  #  5048_0_74_w.png  cos_sim =  0.34324565529823303
AE: throwing out:  15  #  5048_0_79_w.png  cos_sim =  0.43093156814575195
AE: throwing out:  17  #  5048_0_82_w.png  cos_sim =  0.4872162938117981
17
20


In [17]:
def weed_out_odd_images2(embeddings):
    similarity_threshold = 0.60
    num_frames = len(embeddings)
    filenames = img_names_full
    
    # 1. Stack list into a unified tensor (Shape: N, 768)
    # If embeddings is already a stacked tensor, this is a zero-cost operation
    if isinstance(embeddings, list):
        embeddings_tensor = torch.stack(embeddings)
    else:
        embeddings_tensor = embeddings
    
    # 2. Compute the complete all-to-all similarity matrix in one GPU cycle
    # Matrix multiplication of normalized vectors yields the exact cosine similarities
    sim_matrix = torch.mm(embeddings_tensor, embeddings_tensor.t())
    
    # 3. Establish consensus: Count how many other frames each image agrees with
    # We check how many values in each row cross our similarity_threshold requirement
    agreement_mask = sim_matrix >= similarity_threshold
    agreement_counts = agreement_mask.sum(dim=1) # Shape: (N,)
    
    # 4. Filter logic: Keep frames that match the global majority
    # A true frame should agree with at least 50% of the trajectory cohort
    majority_cutoff = num_frames // 2
    
    final_embeddings = []
    for idx in range(num_frames):
        if agreement_counts[idx] >= majority_cutoff:
            #fname = filenames[idx] if idx < len(filenames) else f"Index_{idx}"
            #print(f"AE: [KEPT]: {fname} | Global agreement count: {agreement_counts[idx]}/{num_frames}")
            final_embeddings.append(embeddings_tensor[idx])
        else:
            # Safe localized filename extraction
            fname = filenames[idx] if idx < len(filenames) else f"Index_{idx}"
            print(f"AE: [Vector Purge] throwing out: {fname} | Global agreement count: {agreement_counts[idx]}/{num_frames}")
    
    return final_embeddings

In [18]:
weeded = weed_out_odd_images2(embedded_doors)

AE: [Vector Purge] throwing out: 5048_0_74_w.png | Global agreement count: 3/20
AE: [Vector Purge] throwing out: 5048_0_79_w.png | Global agreement count: 3/20
AE: [Vector Purge] throwing out: 5048_0_82_w.png | Global agreement count: 3/20


In [12]:
#embedded_doors[2].shape
#isinstance(embedded_doors[2], Iterable)
F.cosine_similarity(embedded_full[13], embedded_full[14], dim=0)

for i in range(len(embedded_full) - 1):
    
    img1_d = embedded_doors[i]
    img2_d = embedded_doors[i + 1]
    img1_f = embedded_full[i]
    img2_f = embedded_full[i + 1]
    cos_sim_f = float(F.cosine_similarity(img1_f, img2_f, dim=0))
    cos_sim_d = float(F.cosine_similarity(img1_d, img2_d, dim=0))

    img1_f_name = img_names_full[i]
    img2_f_name = img_names_full[i + 1]
    img1_d_name = img_names_doors_only[i]
    img2_d_name = img_names_doors_only[i + 1]
    
    print(i, " : ", img1_f_name, " vs ", img2_f_name, " = ", cos_sim_f, " ## ", img1_d_name, " vs ", img2_d_name, " = ", cos_sim_d)
    

0  :  5048_0_5_w.png  vs  5048_0_9_w.png  =  0.7327731847763062  ##  5048_0_5.png  vs  5048_0_9.png  =  0.7620337009429932
1  :  5048_0_9_w.png  vs  5048_0_49_w.png  =  0.7327731847763062  ##  5048_0_9.png  vs  5048_0_49.png  =  0.779442310333252
2  :  5048_0_49_w.png  vs  5048_0_52_w.png  =  1.0  ##  5048_0_49.png  vs  5048_0_52.png  =  0.95545494556427
3  :  5048_0_52_w.png  vs  5048_0_54_w.png  =  0.9094258546829224  ##  5048_0_52.png  vs  5048_0_54.png  =  0.9094142317771912
4  :  5048_0_54_w.png  vs  5048_0_58_w.png  =  0.951168954372406  ##  5048_0_54.png  vs  5048_0_58.png  =  0.9419865608215332
5  :  5048_0_58_w.png  vs  5048_0_62_w.png  =  0.9366799592971802  ##  5048_0_58.png  vs  5048_0_62.png  =  0.9398351311683655
6  :  5048_0_62_w.png  vs  5048_0_65_w.png  =  0.6422253847122192  ##  5048_0_62.png  vs  5048_0_65.png  =  0.6344955563545227
7  :  5048_0_65_w.png  vs  5048_0_66_w.png  =  0.5975781679153442  ##  5048_0_65.png  vs  5048_0_66.png  =  0.622123122215271
8  :  5048

In [13]:
import torch
import torch.nn.functional as F

def filter_trajectory_outliers(dino_encoder, image_list, contamination_threshold=0.85):
    """
    Filters out rogue tracking frames by analyzing pairwise cosine similarities
    of their DINOv3 spatial embeddings.
    
    Args:
        dino_encoder: Your active DINOEncoder class instance
        image_list: List of cropped PIL/NumPy door images belonging to the same track_id
        contamination_threshold: Minimum average similarity required to keep a frame
    Returns:
        clean_images: List of images with rogue frames removed
    """
    if len(image_list) <= 2:
        return image_list # Not enough frames to determine an outlier statistically

    # 1. Generate standard visual embeddings for every crop in the track sequence
    # Shape: (N, 768)
    raw_embeddings = dino_encoder.encode_batch(image_list)
    
    # 2. Normalize to unit length for direct cosine matrix multiplication
    norm_embeddings = F.normalize(raw_embeddings, p=2, dim=1)
    
    # 3. Compute the all-to-all pairwise similarity matrix 
    # Shape: (N, N)
    similarity_matrix = torch.mm(norm_embeddings, norm_embeddings.t())
    
    # 4. Calculate the mean agreement score for each frame relative to the group
    # We subtract 1.0 from the sum because every image has a perfect 1.0 similarity with itself
    num_images = len(image_list)
    mean_agreement = (similarity_matrix.sum(dim=1) - 1.0) / (num_images - 1)
    
    # 5. Filter out frames that fall below the semantic consensus
    clean_images = []
    for idx, score in enumerate(mean_agreement.tolist()):
        if score >= contamination_threshold:
            clean_images.append(image_list[idx])
        else:
            print(f"[Outlier Rejected] Frame index {idx} dropped. Agreement score: {score:.4f}")
            
    # Safeguard: If everything got wiped out by a strict threshold, fall back to raw data
    if not clean_images:
        return image_list
        
    return clean_images
